In [0]:
import re
import pandas as pd
from pyspark.sql import functions as F, types as T

# ---- Layer ----
CATALOG = "us_gmsgq_dev"
ALYT    = "gms_us_alyt"        # analytics layer — you have read+write here
MART    = "gms_us_mart"        # read-only for you (source data lives here)

# ---- RAW reference inputs (your uploaded tables) ----
RAW_CLINICAL = f"{CATALOG}.{ALYT}.Clinicalid_deviations"
RAW_DOCS     = f"{CATALOG}.{ALYT}.Documents_numbers_deviations"
RAW_ACRONYM  = f"{CATALOG}.{ALYT}.Acronyms_other_deviations"
RAW_CRO      = f"{CATALOG}.{ALYT}.Cro_deviations"

# ---- SOURCE deviation data (read-only, in mart) ----
SOURCE_TABLE = f"{CATALOG}.{MART}.tw_deviation_data_formatted_rdq"

# ---- OUTPUT lookups (MUST be in analytics layer — you can't write to mart) ----
REF_CLINICAL = f"{CATALOG}.{ALYT}.ref_clinical_norm"
REF_DOCS     = f"{CATALOG}.{ALYT}.ref_docs_norm"
REF_ACRONYM  = f"{CATALOG}.{ALYT}.ref_acronym_norm"
REF_CRO      = f"{CATALOG}.{ALYT}.ref_cro_norm"
REF_UNIFIED  = f"{CATALOG}.{ALYT}.ref_glossary_unified"

# ---- The contract every lookup MUST expose (extra cols allowed) ----
LOOKUP_SCHEMA = ["key_norm", "entity_type", "canonical_id", "enrichment_text"]

# Fuzzy-match acceptance threshold for CRO/system names (0-100 rapidfuzz scale)
CRO_FUZZY_THRESHOLD = 88

print("Config loaded.")
print("  Inputs :", RAW_CLINICAL, RAW_DOCS, RAW_ACRONYM, RAW_CRO, sep="\n           ")
print("  Source :", SOURCE_TABLE)
print("  Output :", REF_UNIFIED)

In [0]:
# ============================================================================
# SECTION A — CLINICAL IDs  (extract + parse; your logic, unchanged)
# ============================================================================
CLINICAL_ID_ALTERNATIVES = [
    r"TAK[-\s_]?\d{2,4}[-_/]\d{3,4}",
    r"TAK[-\s_]?\d{2,4}",
    r"MLN[-\s_]?\d{3,4}[-_/]CCT-\d{2,4}",
    r"MLN[-\s_]?\d{3,4}[-_/]\d{2,4}",
    r"MLN[-\s_]?\d{3,4}",
    r"SHP[-\s_]?\d{3,4}[-_/]\d{2,4}",
    r"SHP[-\s_]?\d{3,4}",
    r"HGT[-\s_]?[A-Z]{2,4}[-_/]\d{2,4}",
    r"CCT[-_]?\d{2,4}",
    r"DEN[-_]?\d{2,4}",
    r"C\d{5}",
]
ID_REGEX = re.compile("|".join(CLINICAL_ID_ALTERNATIVES), flags=re.IGNORECASE)

def _which_prefix(u):
    for p in ("TAK", "MLN", "SHP", "HGT", "CCT", "DEN"):
        if u.startswith(p):
            return p
    if re.match(r"C\d{5}$", u):
        return "INTERNAL"
    return "UNKNOWN"

def parse_clinical_id(raw):
    u = re.sub(r"-{2,}", "-", re.sub(r"[\s_/]+", "-", str(raw).upper())).strip("-")
    prefix = _which_prefix(u)
    compound = suffix = None
    if prefix in ("TAK", "MLN", "SHP"):
        m = re.match(prefix + r"-?(\d+)(?:-(.+))?$", u)
        if m: compound, suffix = m.group(1), m.group(2)
        key = (prefix + "-" + compound) if compound else u
    elif prefix == "HGT":
        m = re.match(r"HGT-([A-Z]{2,4})-(\d+)$", u)
        if m: compound, suffix = m.group(1), m.group(2)
        key = ("HGT-" + compound) if compound else u
    elif prefix in ("CCT", "DEN"):
        m = re.match(prefix + r"-?(\d+)$", u)
        compound, suffix, key = prefix, (m.group(1) if m else None), u
    elif prefix == "INTERNAL":
        m = re.match(r"C(\d+)$", u)
        compound, key = (m.group(1) if m else None), u
    else:
        key = u
    return {"raw": raw, "canonical": u, "prefix": prefix,
            "compound_number": compound, "study_suffix": suffix, "normalized_key": key}

@F.udf(T.ArrayType(T.StringType()))
def extract_clinical_keys_udf(text):
    """Extract clinical IDs from free text, return normalized compound+study keys."""
    if not text: return []
    keys = set()
    for m in ID_REGEX.finditer(str(text)):
        p = parse_clinical_id(m.group(0))
        if p["normalized_key"]: keys.add(p["normalized_key"])
        if p["canonical"]:      keys.add(p["canonical"])
    return list(keys)

@F.udf(T.ArrayType(T.StringType()))
def clinical_keys_from_ref_udf(protocol, dev_name):
    """Reference-side: emit compound + study keys from Protocol Number / Development_Name."""
    keys = set()
    for src in (protocol, dev_name):
        if src and str(src).strip():
            p = parse_clinical_id(str(src))
            if p["normalized_key"]: keys.add(p["normalized_key"])
            if p["canonical"]:      keys.add(p["canonical"])
    return list(keys)

# ============================================================================
# SECTION B — DOCUMENTS  (regex EXACTLY matching your dev_01_doc SQL patterns)
# ============================================================================
# Same prefixes & shapes as your CREATE VIEW: SOP- MTHD- PROC- TOOL- FORM- \bWI-
# Plus SPEC- and MTD- (reference table uses MTD as a legacy variant of MTHD).
DOC_PATTERNS = [
    r"\bSOP-\d+",
    r"\bSPEC-\d+",
    r"\bMTHD-\d+",
    r"\bMTD-\d+",      # legacy method prefix seen in your reference (MTD-002598)
    r"\bPROC-\d+",
    r"\bTOOL-\d+",
    r"\bFORM-\d+",
    r"\bWI-\d+",
]
DOC_REGEX = re.compile("|".join(DOC_PATTERNS), flags=re.IGNORECASE)
DOC_PREFIXES = r"(SOP|SPEC|MTHD|MTD|PROC|TOOL|FORM|WI)"

def norm_doc(v):
    """Canonical doc key. Note: MTD->MTHD collapse so legacy+current align."""
    if not v: return None
    u = re.sub(r"-{2,}", "-", re.sub(r"[\s_]+", "-", str(v).upper())).strip("-")
    if not re.match(DOC_PREFIXES + r"-?\d", u): return None
    u = re.sub(r"^MTD-", "MTHD-", u)          # unify legacy method prefix
    return u

norm_doc_udf = F.udf(norm_doc, T.StringType())

@F.udf(T.ArrayType(T.StringType()))
def extract_doc_keys_udf(text):
    """Extraction-side twin of your SQL: pull all doc IDs from text, normalized."""
    if not text: return []
    out = set()
    for m in DOC_REGEX.finditer(str(text)):
        k = norm_doc(m.group(0))
        if k: out.add(k)
    return list(out)

@F.udf(T.ArrayType(T.StringType()))
def legacy_doc_keys_udf(v):
    """Reference-side: parse messy legacy_numbers__c like
       'LSHIRE_1174436_6_0;;n/a;;TO SOP-0895' -> ['SOP-0895']."""
    if not v: return []
    out = set()
    for chunk in str(v).split(";;"):
        chunk = chunk.strip()
        if not chunk or chunk.lower() == "n/a": continue
        for m in re.finditer(DOC_PREFIXES + r"[-\s]?\d{3,7}", chunk, re.I):
            k = norm_doc(m.group(0))
            if k: out.add(k)
    return list(out)

# ============================================================================
# SECTION C — ACRONYMS  (regex candidate extraction + KNOWN-SET filter)
# ============================================================================
# Pure regex over-extracts (every uppercase token). So: regex to find CANDIDATES,
# then keep only those present in the acronym reference set (built in Cell 3).
# Candidate shapes: 2+ uppercase letters, optional digits, optional (parens).
ACRONYM_CANDIDATE = re.compile(r"\(?[A-Z]{2,}\)?(?:[-/][A-Z0-9]+)?|\([A-Z]{2,}\)\s?[A-Z]{2,}")

def norm_acr(v):
    if not v: return None
    u = re.sub(r"\s+", " ", str(v).strip()).upper()
    return u or None

norm_acr_udf = F.udf(norm_acr, T.StringType())

def make_extract_acronym_udf(known_keys_broadcast):
    """Factory: returns a UDF that extracts only acronyms present in the known set.
       Pass the broadcast set built in Cell 3 (ACR_KEY_SET)."""
    @F.udf(T.ArrayType(T.StringType()))
    def _udf(text):
        if not text: return []
        known = known_keys_broadcast.value
        out = set()
        for m in ACRONYM_CANDIDATE.finditer(str(text)):
            k = norm_acr(m.group(0))
            if k and k in known:
                out.add(k)
        return list(out)
    return _udf

# ============================================================================
# SECTION D — CRO / SYSTEMS  (normalize only; matching is FUZZY at join time)
# ============================================================================
def norm_name(v):
    if not v: return None
    u = re.sub(r"[^A-Z0-9 ]", " ", str(v).upper())
    u = re.sub(r"\s+", " ", u).strip()
    return u or None

norm_name_udf = F.udf(norm_name, T.StringType())

@F.udf(T.ArrayType(T.StringType()))
def cro_variants_udf(name, disp, alias):
    """Reference-side: normalized name/displayName/alias variants for fuzzy pool."""
    return list({norm_name(x) for x in (name, disp, alias) if x and norm_name(x)})

print("Shared UDFs registered: clinical, doc, acronym(factory), cro.")

In [0]:
ac = spark.table(RAW_ACRONYM)

# Guard: don't let short acronyms shadow real clinical/doc IDs
ID_LIKE = r"^(TAK|MLN|SHP|HGT|CCT|DEN|SOP|SPEC|MTHD|FORM|TOOL|WI|PROC)[-\s]?\d"

ref_acronym = (
    ac.withColumn("key_norm", norm_acr_udf(F.col("Acronym")))
      .filter(F.col("key_norm").isNotNull())
      .filter(F.length("key_norm") >= 2)                      # drop 1-char noise
      .filter(~F.col("key_norm").rlike(ID_LIKE))              # don't shadow IDs
      .filter(F.col("Category").isin("Medical", "Industry"))  # keep relevant senses
      .groupBy("key_norm")
      .agg(
          F.concat_ws(" | ", F.collect_set("Definition")).alias("senses"),
          F.first("Acronym", ignorenulls=True).alias("canonical_id"),
      )
      .withColumn("entity_type", F.lit("ACRONYM"))
      .withColumn("enrichment_text", F.concat(F.lit("Acronym expansions: "), F.col("senses")))
      .select(*LOOKUP_SCHEMA)
)

(ref_acronym.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_ACRONYM))
spark.sql(f"COMMENT ON TABLE {REF_ACRONYM} IS "
          "'Sense-aware acronym lookup: key_norm→all definitions. Built from raw_acronym.'")

display(spark.table(REF_ACRONYM).limit(20))

# Build a plain dict {ACRONYM_UPPER: first_definition} for TA/modality expansion in Cell 4
_acr_pd = (ac.filter(F.col("Category").isin("Medical", "Industry"))
             .select("Acronym", "Definition").toPandas())
ACR_MAP = {norm_acr(a): d for a, d in zip(_acr_pd["Acronym"], _acr_pd["Definition"]) if norm_acr(a)}

# Hand-curated TA overrides win over the generic acronym table
TA_OVERRIDES = {
    "NS": "Neuroscience", "ONC": "Oncology", "GI": "Gastroenterology",
    "RARE": "Rare Diseases", "PDT": "Plasma-Derived Therapies",
}
def expand_abbr(v):
    if v is None or str(v).strip() == "":
        return ""
    u = norm_acr(v)
    return TA_OVERRIDES.get(u) or ACR_MAP.get(u) or v   # fall back to original text
expand_abbr_udf = F.udf(expand_abbr, T.StringType())

print(f"Acronym map size: {len(ACR_MAP):,}")

In [0]:
# (add to end of Cell 3, after ref_acronym is built)
acr_key_set = set(r["key_norm"] for r in spark.table(REF_ACRONYM).select("key_norm").collect())
ACR_KEY_SET = spark.sparkContext.broadcast(acr_key_set)
extract_acronym_keys_udf = make_extract_acronym_udf(ACR_KEY_SET)
print(f"Acronym known-set size: {len(acr_key_set):,}")

In [0]:
clin = spark.table(RAW_CLINICAL)

# NOTE: your raw columns have spaces/parentheses. Wrap those in backticks when referenced.
clin_enriched = clin.withColumn(
    "enrichment_text",
    F.concat_ws(
        "\n",
        F.concat(F.lit("Generic: "),   F.coalesce(F.col("Generic_Name"), F.lit(""))),
        F.concat(F.lit("Brand: "),     F.coalesce(F.col("Brand_Name"), F.lit(""))),
        F.concat(F.lit("Modality: "),  expand_abbr_udf(F.col("Modality")),
                 F.lit(" "),           F.coalesce(F.col("Modality_Detail"), F.lit(""))),
        F.concat(F.lit("Mechanism: "), F.coalesce(F.col("Mechanism"), F.lit(""))),
        F.concat(F.lit("Target: "),    F.coalesce(F.col("Target_Long_Name"), F.lit(""))),
        F.concat(F.lit("Therapeutic Area: "), expand_abbr_udf(F.col("PF_TherapeuticArea"))),
        F.concat(F.lit("Indication: "), F.coalesce(F.col("IND_DESC"), F.lit(""))),
        F.concat(F.lit("Phase: "),     F.coalesce(F.col("`Phase (Nominal)`"), F.lit(""))),
    ),
)

ref_clinical = (
    clin_enriched
    .withColumn("key_norm", F.explode(
        clinical_keys_udf(F.col("`Protocol Number`"), F.col("Development_Name"))))
    .filter(F.col("key_norm").isNotNull() & (F.col("key_norm") != ""))
    .withColumn("entity_type", F.lit("CLINICAL_ID"))
    .withColumn("canonical_id", F.coalesce(F.col("Development_Name"), F.col("`Protocol Number`")))
    .select(*LOOKUP_SCHEMA)
    .dropDuplicates(["key_norm"])
)

(ref_clinical.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_CLINICAL))
spark.sql(f"COMMENT ON TABLE {REF_CLINICAL} IS "
          "'Clinical-ID lookup (compound+study keys); TA/modality expanded via acronym map. From raw_clinical_ref.'")

display(spark.table(REF_CLINICAL).limit(20))